# SQL query from table names - Continued

In [20]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [21]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [22]:
context = [ {'role':'system', 'content':"""
    ###TASK
    Generate a SQL query to answer the question, given a SQL database schema.
             
    ###INSTRUCTIONS
    - Deliberately go through the question and database schema word by word** to appropriately answer the question.
    - if you cannot answer the question with the available database schema, return "I don't know".
    - Your Database is composed by a SQL database with some tables.
    - Try to maintain the SQL order simple.
        
             
    ###SCHEMA
    CREATE TABLE employees (
        ID_usr INTEGER,      -- unique ID of the employee
        name VARCHAR(30)     -- name of the employee
    )

    CREATE TABLE salary (
        ID_usr INTEGER,      -- ID of the employee (links to employees.ID_usr)
        year DATE,           -- year of the salary record
        salary FLOAT         -- annual salary amount
    )

    CREATE TABLE studies (
        ID INTEGER,                -- unique ID of the study record
        ID_usr INTEGER,            -- ID of the employee (links to employees.ID_usr)
        educational_level INTEGER, -- level of education
        Institution VARCHAR(30),   -- name of the institution
        Years DATE,                -- years of study
        Speciality VARCHAR(30)     -- field of specialization
    )

    -- employees.ID_usr can be joined with salary.ID_usr
    -- employees.ID_usr can be joined with studies.ID_usr
        
             
    ###ANSWER
    Given the database schema, here is the SQL query that answers the question.
             """} ]



In [23]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
Question: Salary evolution over time for a specific employee
SELECT e.name, s.year, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE e.name = 'John Doe'
ORDER BY s.year ASC
Question: Count the number of employees per speciality
SELECT st.Speciality, COUNT(e.ID_usr) AS num_employees
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
GROUP BY st.Speciality
ORDER BY num_employees DESC
"""
})

In [24]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [25]:
#new
context_user = context.copy()
print(return_CCRMSQL("Who is the employee with highest salary?", context_user))

I don't know.


In [26]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Who is the employee with highest salary?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of the employee with the highest salary by joining the "employees" table with the "salary" table on the ID_usr column. It then orders the result by salary in descending order and limits the output to only one row, which corresponds to the employee with the highest salary.


In [27]:
#new
print(return_CCRMSQL("What is the institution with the highest average salary?", context_user))

I don't know.


In [28]:
#old
print(return_CCRMSQL("What is the institution with the highest average salary?", old_context_user))

This is your SQL:
```sql
SELECT s.Institution, AVG(sa.salary) AS avg_salary
FROM studies s
JOIN employees e ON s.ID_usr = e.ID_usr
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query joins the tables `studies`, `employees`, and `salary` based on the user ID. It calculates the average salary for each institution by grouping the results by institution. Then, it orders the results in descending order of average salary and limits the output to show only the institution with the highest average salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [30]:
context_user = [{'role': 'system', 'content':"""
###TASK
Generate a SQL query to answer the question, given the following SQL database schema.

###INSTRUCTIONS
- If you cannot answer the question with the available database schema, return 'I do not know'
- The older employees by age are the ones that display higher number of employee_age
- An employee should be linked to one and only one store_ID
- Years of seniority is stored in employee_yearsinthecompany as an integer


###SCHEMA
CREATE TABLE employees(
    employee_ID INTEGER PRIMARY KEY, -- Unique ID for each employee
    employee_age INTEGER, -- age of the employee
    employee_name VARCHAR(50), -- Name of the employee
    employee_yearsinthecompany INTEGER, -- number of years of the employee in the company
    store_ID INTEGER -- ID of stores where the employee work
);

CREATE TABLE stores(
    store_ID INTEGER PRIMARY KEY, -- Unique ID for each store
    store_state VARCHAR(50), -- State name where the store is located
    store_city VARCHAR(50), -- City name where the store is located
    store_address VARCHAR(500) -- Street name where the store is located
);

- stores.store_ID can be joined with employees.store_ID


###ANSWER
Given the database schema, here is the SQL query that answers quesiton.

"""}]
question = "Who are the 10 oldest employees by age in New York city?"
print(return_CCRMSQL(question, context_user))

I do not know


In [31]:
context = [{'role': 'system', 'content':"""
#TASK
Generate a SQL query to answer the question, given the following SQL database schema.

#INSTRUCTIONS
- If you cannot answer the question with the available database schema, return 'I do not know'
- The fastest lap per circuit is the lap with the minimum lap_time value
- lap_time is stored in seconds as a DECIMAL
- A driver can have multiple laps across multiple circuits

#SCHEMA
CREATE TABLE circuits (
circuit_ID INTEGER PRIMARY KEY, -- Unique ID for each circuit
circuit_name VARCHAR(100) -- Name of the circuit
)

CREATE TABLE laps (
lap_ID INTEGER PRIMARY KEY, -- Unique ID for each lap record
circuit_ID INTEGER, -- ID of the circuit where the lap was driven
lap_number INTEGER, -- Lap number within the race
lap_time DECIMAL(8,3), -- Lap time in seconds
driver_ID INTEGER -- ID of the driver who drove the lap
);

CREATE TABLE drivers (
driver_ID INTEGER PRIMARY KEY, -- Unique ID for each driver
driver_name VARCHAR(50), -- name of the drivers
driver_age INTEGER -- age of the drivers
)

-- laps.circuit_ID can be joined with circuits.circuit_ID
-- laps.driver_ID can be joined with drivers.driver_ID



#ANSWER
Given the database schema, here is the SQL query that answers quesiton.
"""}]
context_user = context.copy()
question = "What is the age of the driver with the fastest lap on each circuit?"
print(return_CCRMSQL(question, context_user))

```sql
SELECT c.circuit_name, d.driver_age
FROM circuits c
JOIN (
    SELECT l.circuit_ID, MIN(l.lap_time) AS fastest_lap_time, d.driver_age
    FROM laps l
    JOIN drivers d ON l.driver_ID = d.driver_ID
    GROUP BY l.circuit_ID
) AS fastest_lap_per_circuit ON c.circuit_ID = fastest_lap_per_circuit.circuit_ID
JOIN drivers d ON fastest_lap_per_circuit.driver_age = d.driver_age;
```


In [32]:
context = [{'role': 'system', 'content':"""
### TASK
Generate a SQL query to answer the question, given the following SQL database schema.

### INSTRUCTIONS
- If you cannot answer the question with the available database schema, return 'I do not know'
- Top scorers are ranked by the total number of goals they scored (goals_scored column in match_events)
- Only count goals of type 'goal' — exclude own goals (type = 'own_goal') from individual tallies
- A player belongs to one team; a team participates in many games

### SCHEMA
CREATE TABLE games (
  game_ID INTEGER PRIMARY KEY, -- Unique ID for each game
  home_team_ID VARCHAR(5), -- Team ID of the home team
  away_team_ID VARCHAR(5), -- Team ID of the away team
  game_date DATE, -- Date the game was played
  home_score INTEGER, -- Final score of the home team
  away_score INTEGER -- Final score of the away team
);

CREATE TABLE teams (
  team_ID VARCHAR(5) PRIMARY KEY, -- Unique short ID for each team
  team_name VARCHAR(100) -- Full name of the team
);

CREATE TABLE players (
  player_ID INTEGER PRIMARY KEY, -- Unique ID for each player
  player_name VARCHAR(100), -- Full name of the player
  team_ID VARCHAR(5), -- Team the player belongs to
  position VARCHAR(30) -- Playing position (e.g. Forward, Midfielder)
);

CREATE TABLE match_events (
event_ID INTEGER PRIMARY KEY, -- Unique ID for each event
game_ID INTEGER, -- ID of the game where the event occurred
player_ID INTEGER, -- ID of the player who triggered the event
event_type VARCHAR(20), -- Type of event: 'goal', 'own_goal', 'yellow_card', 'red_card'
event_minute INTEGER -- Minute of the match when the event occurred
);

-- games.home_team_ID and games.away_team_ID can be joined with teams.team_ID
-- players.team_ID can be joined with teams.team_ID
-- match_events.game_ID can be joined with games.game_ID
-- match_events.player_ID can be joined with players.player_ID

### ANSWER
Given the database schema, here is the SQL query that answers quesiton.
"""
}]
context_user = context.copy()
question = "Who are the top scorers of the championship?"
print(return_CCRMSQL(question, context_user))

```sql
SELECT p.player_name, COUNT(*) AS total_goals
FROM players p
JOIN match_events me ON p.player_ID = me.player_ID
WHERE me.event_type = 'goal'
GROUP BY p.player_name
ORDER BY total_goals DESC;
```


Comment:
- Variation 1 — The results "I don't know" seems a hallucinated refusal for a perfectly answerable query. Maybe due to the prompt included the fallback instruction "return I do not know" and adding to the fact had no few-shot examples to show the model what an answerable query looks like.
- Variation 2 — The results is partially correct SQL as the model used the right structural approach but made a subtle semantic error: it joined on driver_age instead of driver_ID.
- Variation 3 — The results is accurate SQL. The own-goal filter (event_type = 'goal') was correctly applied, the join was right, and the aggregation as well.